# ETL Transformación - De Staging a DWH

Este notebook realiza la transformación de datos desde las tablas staging (STG_Universidad) hacia el Data Warehouse (dw_universidad).

## Procesos incluidos:
- **Limpieza**: Reparación de encoding, espacios, valores nulos
- **Validación**: Tipos de datos, rangos, integridad referencial
- **Normalización**: Formatos de fechas, mayúsculas, decimales
- **Deduplicación**: Eliminación de registros duplicados
- **Optimización**: Procesamiento por lotes y transacciones

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text, MetaData, Table
from sqlalchemy.pool import NullPool
from datetime import datetime, date
from dotenv import load_dotenv
import os
from pathlib import Path
import warnings
import re
from typing import Tuple, Dict, List
import sys

warnings.filterwarnings('ignore')

# Agregar ruta del proyecto al path para importar módulos
sys.path.append(os.path.join(os.getcwd(), '..'))

# Importar el LoggerManager
from logging_config import LoggerManager

# ============================================
# CONFIGURACIÓN DE CREDENCIALES Y CONEXIÓN
# ============================================

load_dotenv()

USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
STG_DATABASE = os.getenv("STG_DATABASE")
DWH_DATABASE = os.getenv("DWH_DATABASE")

# Motor para Staging (lectura)
engine_stg = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{STG_DATABASE}",
    poolclass=NullPool
)

# Motor para DWH (escritura)
engine_dwh = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DWH_DATABASE}",
    poolclass=NullPool
)

print("[INFO] Motores de conexión configurados")
print(f"  Staging: {STG_DATABASE} en {HOST}:{PORT}")
print(f"  DWH: {DWH_DATABASE} en {HOST}:{PORT}")

# Configurar logger para este proceso (logs en carpeta actual: 3-ETL_CargaInicial/logs)
logger = LoggerManager.configurar("transformacion", ruta_raiz=os.getcwd(), carpeta_logs='logs')

# Obtener ruta de logs (creada automáticamente por LoggerManager)
RUTA_LOGS = LoggerManager.obtener_ruta_logs()

print(f"\n[OK] Logging configurado en: {RUTA_LOGS}")

2026-05-11 03:24:20 - INFO - Iniciando transformacion. Log: c:\Users\maria\OneDrive\Documentos\GitHub\ADE2026_TpiUniversidad\TP2\2-ETL_CargaInicial\logs\transformacion_20260511_032420.log


[INFO] Motores de conexión configurados
  Staging: stg_universidad en localhost:3306
  DWH: dw_universidad en localhost:3306

[OK] Logging configurado en: c:\Users\maria\OneDrive\Documentos\GitHub\ADE2026_TpiUniversidad\TP2\2-ETL_CargaInicial\logs


In [2]:
# ============================================
# FUNCIONES DE LIMPIEZA Y VALIDACIÓN
# ============================================

class DataCleaner:
    """Clase para limpieza y validación de datos."""
    
    @staticmethod
    def limpiar_string(valor: str) -> str:
        """Limpia strings: espacios, codificación, minúsculas."""
        if pd.isna(valor) or valor is None:
            return None
        
        # Convertir a string si no lo es
        valor = str(valor).strip()
        
        if valor.lower() in ['', 'null', 'none', 'n/a', 'sin dato']:
            return None
        
        # Reparar encoding (caracteres corruptos por UTF-8/Latin-1)
        try:
            valor = valor.encode('latin-1').decode('utf-8', errors='ignore')
        except:
            pass
        
        return valor
    
    @staticmethod
    def limpiar_numero(valor, tipo='float', requerido=False) -> any:
        """Convierte y valida números."""
        if pd.isna(valor) or valor is None:
            if requerido:
                LoggerManager.warning("Valor requerido faltante")
            return None
        
        valor_str = str(valor).strip()
        
        if valor_str.lower() in ['', 'null', 'none', 'n/a']:
            return None
        
        try:
            valor_str = valor_str.replace(',', '.').replace(' ', '')
            
            if tipo == 'int':
                return int(float(valor_str))
            elif tipo == 'float':
                return float(valor_str)
            else:
                return None
        except:
            LoggerManager.warning(f"No se pudo convertir '{valor}' a {tipo}")
            return None
    
    @staticmethod
    def limpiar_fecha(valor) -> date:
        """Convierte y valida fechas (múltiples formatos)."""
        if pd.isna(valor) or valor is None:
            return None
        
        valor_str = str(valor).strip()
        
        if valor_str.lower() in ['', 'null', 'none', 'n/a']:
            return None
        
        formatos = [
            '%Y-%m-%d',
            '%d/%m/%Y',
            '%Y%m%d',
            '%d-%m-%Y',
            '%m-%d-%Y',
            '%Y',
            '%d/%m/%y',
            '%Y/%m/%d',
        ]
        
        for fmt in formatos:
            try:
                return pd.to_datetime(valor_str, format=fmt).date()
            except:
                continue
        
        try:
            return pd.to_datetime(valor_str).date()
        except:
            LoggerManager.warning(f"No se pudo parsear fecha: '{valor}'")
            return None
    
    @staticmethod
    def limpiar_genero(valor: str) -> str:
        """Normaliza valores de género."""
        if pd.isna(valor) or valor is None:
            return None
        
        valor = str(valor).strip().upper()
        
        if valor in ['M', 'MASCULINO', 'MALE', 'HOMBRE','1']:
            return 'M'
        elif valor in ['F', 'FEMENINO', 'FEMALE', 'MUJER','2']:
            return 'F'
        else:
            LoggerManager.warning(f"Género desconocido: '{valor}'")
            return None
    
    @staticmethod
    def limpiar_nombre_duplicado(nombre: str) -> str:
        """Elimina palabras duplicadas consecutivas en nombres (ej: 'Carlos Carlos' -> 'Carlos')."""
        if pd.isna(nombre) or nombre is None:
            return None
        
        nombre = str(nombre).strip()
        
        if not nombre:
            return None
        
        palabras = nombre.split()
        
        if len(palabras) == 0:
            return None
        
        palabras_limpias = [palabras[0]]
        for palabra in palabras[1:]:
            if palabra.lower() != palabras_limpias[-1].lower():
                palabras_limpias.append(palabra)
        
        return ' '.join(palabras_limpias)
    
    @staticmethod
    def limpiar_codigo_identificador(valor: str) -> str:
        """Limpia códigos de identificación (DNI, códigos de curso) como strings.
        No convierte a número, solo quita espacios y caracteres especiales."""
        if pd.isna(valor) or valor is None:
            return None
        
        valor = str(valor).strip()
        
        if valor.lower() in ['', 'null', 'none', 'n/a', 'sin dato']:
            return None
        
        # Para códigos: remover espacios, puntos y guiones, pero mantener como string
        valor = valor.replace(' ', '').replace('.', '').replace('-', '')
        
        # Si el resultado está vacío después de limpiar, retornar None
        if not valor:
            return None
        
        return valor
    
    @staticmethod
    def validar_dni(dni) -> bool:
        """Valida rango de DNI argentino (7 a 8 dígitos) sin convertir a número.
        Recibe el DNI como string y valida que solo contenga dígitos en rango válido."""
        if pd.isna(dni) or dni is None:
            return False
        
        try:
            # Convertir a string si no lo es
            dni_str = str(dni).strip().replace(' ', '').replace('.', '').replace('-', '')
            
            # Verificar que solo contiene dígitos
            if not dni_str.isdigit():
                return False
            
            # Convertir solo para validar rango (no se modifica el original)
            dni_num = int(dni_str)
            
            # Validar que esté en el rango legal (7 a 8 dígitos)
            return 1000000 <= dni_num <= 99999999
            
        except (ValueError, TypeError, AttributeError):
            return False
        
    @staticmethod
    def limpiar_nacionalidad(valor: str) -> str:
        """Normaliza valores de nacionalidad."""
        if pd.isna(valor) or valor is None:
            return 'No especificado'
        return str(valor).strip()
    
    @staticmethod
    def validar_nota(nota: float) -> bool:
        """Valida rango de nota (0-10)."""
        if pd.isna(nota) or nota is None:
            return False
        return 0 <= nota <= 10

print("[OK] Clase DataCleaner cargada")

[OK] Clase DataCleaner cargada


In [3]:
def transformar_estudiante(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """
    Transforma datos de STG_ESTUDIANTE a formato DWH.
    
    Limpieza:
    - DNI como string (sin convertir a número)
    - Nombres sin duplicación de palabras
    - Validación de rango de DNI
    """
    cleaner = DataCleaner()
    stats = {'total': len(df), 'válidos': 0, 'rechazados': 0, 'duplicados': 0}

    # Limpiar columnas básicas
    df['id_estudiante'] = df['id_estudiante_raw'].apply(
        lambda x: cleaner.limpiar_numero(x, 'int')
    )
    # DNI como STRING, no número
    df['dni'] = df['dni_raw'].apply(lambda x: cleaner.limpiar_codigo_identificador(x))
    df['apellido'] = df['apellido_raw'].apply(lambda x: cleaner.limpiar_string(x))
    df['apellido'] = df['apellido'].apply(lambda x: cleaner.limpiar_nombre_duplicado(x))
    df['nombre'] = df['nombre_raw'].apply(lambda x: cleaner.limpiar_string(x))
    df['nombre'] = df['nombre'].apply(lambda x: cleaner.limpiar_nombre_duplicado(x))
    df['genero'] = df['genero_raw'].apply(lambda x: cleaner.limpiar_genero(x))
    df['fecha_nacimiento'] = df['fecha_nacimiento_raw'].apply(
        lambda x: cleaner.limpiar_fecha(x)
    )
    df['email'] = df['email_raw'].apply(lambda x: cleaner.limpiar_string(x))
    df['nacionalidad'] = df['nacionalidad_raw'].apply(lambda x: cleaner.limpiar_nacionalidad(x))
    df['id_programa'] = df['id_programa_raw'].apply(
        lambda x: cleaner.limpiar_numero(x, 'int')
    )
    df['anio_ingreso'] = df['anio_ingreso_raw'].apply(
        lambda x: cleaner.limpiar_numero(x, 'int')
    )

    # Campos por defecto para SCD
    df['valid_from'] = date.today()
    df['valid_to'] = None
    df['es_actual'] = True
    df['egreso_carrera'] = False
    df['anio_egreso'] = None
    df['abandono_carrera'] = False
    df['anio_abandono'] = None

    # Validaciones
    df['es_válido'] = df.apply(
        lambda row: (
            row['id_estudiante'] is not None and
            row['dni'] is not None and
            cleaner.validar_dni(row['dni']) and
            row['apellido'] is not None and
            row['nombre'] is not None and
            row['id_programa'] is not None
        ),
        axis=1
    )

    # Separar válidos de inválidos
    df_válidos = df[df['es_válido']].copy()
    df_inválidos = df[~df['es_válido']].copy()

    stats['válidos'] = len(df_válidos)
    stats['rechazados'] = len(df_inválidos)

    if len(df_inválidos) > 0:
        LoggerManager.warning(f"Estudiantes rechazados: {len(df_inválidos)}")

    # Deduplicación por DNI
    duplicados = df_válidos[df_válidos.duplicated(subset=['dni'], keep=False)]
    stats['duplicados'] = len(duplicados)

    if len(duplicados) > 0:
        LoggerManager.warning(f"Registros con DNI duplicado: {len(duplicados)}")
        df_válidos = df_válidos.drop_duplicates(subset=['dni'], keep='first')

    # Seleccionar columnas
    columnas = [
        'id_estudiante', 'dni', 'apellido', 'nombre', 'genero',
        'fecha_nacimiento', 'nacionalidad', 'anio_ingreso',
        'egreso_carrera', 'anio_egreso', 'abandono_carrera', 'anio_abandono',
        'valid_from', 'valid_to', 'es_actual'
    ]

    return df_válidos[columnas], stats


def transformar_dictado(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """
    Transforma datos de STG_DICTADO a formato DWH.
    
    Limpieza:
    - Códigos de curso como STRING
    - Validación básica de campos requeridos
    """
    cleaner = DataCleaner()
    stats = {'total': len(df), 'válidos': 0, 'rechazados': 0, 'duplicados': 0}

    # Limpiar columnas básicas
    df['id_dictado'] = df['id_dictado_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    df['id_curso'] = df['id_curso_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    df['id_docente'] = df['id_docente_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    df['id_programa'] = df['id_programa_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    df['periodo'] = df['periodo_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    df['turno'] = df['turno_raw'].apply(lambda x: cleaner.limpiar_string(x))
    df['aula'] = df['aula_raw'].apply(lambda x: cleaner.limpiar_string(x))
    df['cupo_max'] = df['cupo_maximo_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    
    # Código de curso como STRING
    df['codigo_curso'] = df['codigo_raw'].apply(lambda x: cleaner.limpiar_codigo_identificador(x))

    # Campos por defecto para SCD
    df['valid_from'] = date.today()
    df['valid_to'] = None
    df['es_actual'] = True

    # Validaciones
    df['es_válido'] = df.apply(
        lambda row: (
            row['id_dictado'] is not None and
            row['id_curso'] is not None and
            row['id_docente'] is not None
        ),
        axis=1
    )

    df_válidos = df[df['es_válido']].copy()
    stats['válidos'] = len(df_válidos)
    stats['rechazados'] = len(df) - len(df_válidos)

    # Deduplicación
    df_válidos = df_válidos.drop_duplicates(subset=['id_dictado'], keep='first')
    stats['duplicados'] = stats['válidos'] - len(df_válidos)

    # Seleccionar columnas
    columnas = [
        'id_dictado', 'codigo_curso', 'periodo', 'turno', 'aula', 'cupo_max',
        'id_curso', 'id_docente', 'id_programa',
        'valid_from', 'valid_to', 'es_actual'
    ]

    return df_válidos[columnas], stats


def transformar_inscripcion(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """
    Transforma datos de STG_INSCRIPCION a fact_inscripcion.
    """
    cleaner = DataCleaner()
    stats = {'total': len(df), 'válidos': 0, 'rechazados': 0, 'duplicados': 0}

    df['id_inscripcion'] = df['id_inscripcion_raw'].apply(
        lambda x: cleaner.limpiar_numero(x, 'int')
    )
    df['id_estudiante'] = df['id_estudiante_raw'].apply(
        lambda x: cleaner.limpiar_numero(x, 'int')
    )
    df['id_dictado'] = df['id_dictado_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    df['fecha_inscripcion'] = df['fecha_inscripcion_raw'].apply(
        lambda x: cleaner.limpiar_fecha(x)
    )
    df['estado'] = df['estado_raw'].apply(lambda x: cleaner.limpiar_string(x))
    df['tiempo_skey'] = df['fecha_inscripcion'].apply(
        lambda x: int(x.strftime('%Y%m%d')) if x else None
    )
    df['abandono'] = False

    df['es_válido'] = df.apply(
        lambda row: (
            row['id_inscripcion'] is not None and
            row['id_estudiante'] is not None and
            row['id_dictado'] is not None and
            row['tiempo_skey'] is not None
        ),
        axis=1
    )

    df_válidos = df[df['es_válido']].copy()
    stats['válidos'] = len(df_válidos)
    stats['rechazados'] = len(df) - len(df_válidos)

    df_válidos = df_válidos.drop_duplicates(subset=['id_inscripcion'], keep='first')
    stats['duplicados'] = stats['válidos'] - len(df_válidos)

    columnas = ['id_estudiante', 'tiempo_skey', 'id_dictado', 'estado', 'abandono']

    return df_válidos[columnas], stats


def transformar_examen(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """
    Transforma datos de STG_EXAMEN a fact_examen_estudiante.
    Incluye validación de consistencia entre resultado y nota (>= 6 es aprobado).
    """
    cleaner = DataCleaner()
    stats = {'total': len(df), 'válidos': 0, 'rechazados': 0, 'duplicados': 0}

    df['id_examen'] = df['id_examen_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    df['id_inscripcion'] = df['id_inscripcion_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    df['fecha'] = df['fecha_raw'].apply(lambda x: cleaner.limpiar_fecha(x))
    df['nota'] = df['nota_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'float'))
    df['numero_intento'] = df['numero_intento_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    df['resultado'] = df['resultado_raw'].apply(lambda x: cleaner.limpiar_string(x))

    # Normalizar resultado
    df['resultado'] = df['resultado'].apply(
        lambda x: 'Aprobado' if x and x.lower() in ['aprobado', 'aprob', 'sí', 'si', '1']
        else 'Desaprobado' if x else None
    )

    # Calcular aprobado basado en nota (nota >= 6 es aprobado)
    df['aprobado'] = df['nota'].apply(lambda x: x >= 6 if x is not None else None)
    
    # Validar consistencia entre resultado y nota
    df['resultado_consistente'] = df.apply(
        lambda row: (
            (row['aprobado'] and row['resultado'] == 'Aprobado') or
            (not row['aprobado'] and row['resultado'] == 'Desaprobado')
        ) if row['aprobado'] is not None and row['resultado'] is not None else True,
        axis=1
    )
    
    # Marcar inconsistencias
    inconsistencias = df[~df['resultado_consistente']]
    if len(inconsistencias) > 0:
        LoggerManager.warning(f"Inconsistencias entre resultado y nota en examen: {len(inconsistencias)} registros")
        for idx, row in inconsistencias.iterrows():
            LoggerManager.warning(f"ID Examen {row['id_examen']}: Nota={row['nota']}, Resultado={row['resultado']}, Aprobado={row['aprobado']}")

    df['tiempo_skey'] = df['fecha'].apply(
        lambda x: int(x.strftime('%Y%m%d')) if x else None
    )

    df['es_válido'] = df.apply(
        lambda row: (
            row['id_examen'] is not None and
            row['id_inscripcion'] is not None and
            row['nota'] is not None and
            cleaner.validar_nota(row['nota']) and
            row['numero_intento'] is not None and
            row['numero_intento'] > 0 and
            row['fecha'] is not None
        ),
        axis=1
    )

    df_válidos = df[df['es_válido']].copy()
    stats['válidos'] = len(df_válidos)
    stats['rechazados'] = len(df) - len(df_válidos)

    df_válidos = df_válidos.drop_duplicates(subset=['id_examen'], keep='first')
    stats['duplicados'] = stats['válidos'] - len(df_válidos)

    columnas = ['id_inscripcion', 'tiempo_skey', 'nota', 'numero_intento', 'aprobado']

    return df_válidos[columnas], stats


def transformar_evaluacion_curso(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """
    Transforma datos de STG_EVALUACION_CURSO a fact_evaluacion_dictado.
    """
    cleaner = DataCleaner()
    stats = {'total': len(df), 'válidos': 0, 'rechazados': 0, 'duplicados': 0}

    df['id_evaluacion'] = df['id_evaluacion_raw'].apply(
        lambda x: cleaner.limpiar_numero(x, 'int')
    )
    df['id_dictado'] = df['id_dictado_raw'].apply(lambda x: cleaner.limpiar_numero(x, 'int'))
    df['fecha_evaluacion'] = df['fecha_evaluacion_raw'].apply(
        lambda x: cleaner.limpiar_fecha(x)
    )
    df['puntaje_dictado'] = df['puntaje_dictado_raw'].apply(
        lambda x: cleaner.limpiar_numero(x, 'float')
    )
    df['puntaje_contenido'] = df['puntaje_contenido_raw'].apply(
        lambda x: cleaner.limpiar_numero(x, 'float')
    )
    df['valoracion_general'] = df['valoracion_general_raw'].apply(
        lambda x: cleaner.limpiar_numero(x, 'float')
    )

    df['tiempo_skey'] = df['fecha_evaluacion'].apply(
        lambda x: int(x.strftime('%Y%m%d')) if x else None
    )

    df['nota_general'] = df[['puntaje_dictado', 'puntaje_contenido', 'valoracion_general']].mean(axis=1)

    df['es_válido'] = df.apply(
        lambda row: row['id_evaluacion'] is not None and row['id_dictado'] is not None,
        axis=1
    )

    df_válidos = df[df['es_válido']].copy()
    stats['válidos'] = len(df_válidos)
    stats['rechazados'] = len(df) - len(df_válidos)

    df_válidos = df_válidos.drop_duplicates(subset=['id_evaluacion'], keep='first')
    stats['duplicados'] = stats['válidos'] - len(df_válidos)

    columnas = ['id_dictado', 'tiempo_skey', 'puntaje_dictado', 'puntaje_contenido', 'nota_general']

    return df_válidos[columnas], stats


def cargar_dim_tiempo() -> Dict:
    """Carga la dimensión temporal (dim_tiempo) con fechas desde 2015 hasta 2030."""
    from datetime import datetime, date, timedelta

    fechas = []
    fecha_actual = date(2015, 1, 1)
    fin = date(2030, 12, 31)

    while fecha_actual <= fin:
        fechas.append({
            'tiempo_skey': int(fecha_actual.strftime('%Y%m%d')),
            'fecha': fecha_actual,
            'dia': fecha_actual.day,
            'mes': fecha_actual.strftime('%B'),
            'anio': fecha_actual.year,
            'periodo_academico': '1er Cuatrimestre' if fecha_actual.month <= 6 else '2do Cuatrimestre',
            'es_feriado': False
        })
        fecha_actual = fecha_actual + timedelta(days=1)

    df_tiempo = pd.DataFrame(fechas)

    stats_insert = insertar_en_lotes(df_tiempo, 'dim_tiempo', tamaño_lote=1000)
    LoggerManager.info(f"Dimensión tiempo cargada: {stats_insert['insertados']} fechas")

    return stats_insert


def insertar_en_lotes(df: pd.DataFrame, tabla_destino: str, tamaño_lote: int = 500) -> Dict:
    """Inserta datos en el DWH en lotes con manejo de transacciones."""
    if df.empty:
        LoggerManager.warning(f"DataFrame vacío para {tabla_destino}")
        return {'insertados': 0, 'errores': 0, 'lotes': 0}
    
    resultados = {'insertados': 0, 'errores': 0, 'lotes': 0}
    
    num_lotes = (len(df) + tamaño_lote - 1) // tamaño_lote
    
    try:
        if tabla_destino == 'dim_tiempo':
            # dim_tiempo tiene foreign keys, desactivar restricciones temporalmente
            with engine_dwh.begin() as connection:
                connection.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
                connection.execute(text(f"DELETE FROM {tabla_destino}"))
                connection.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
                LoggerManager.info(f"DELETE: {tabla_destino} (con FK desactivadas)")
        else:
            with engine_dwh.begin() as connection:
                connection.execute(text(f"TRUNCATE TABLE {tabla_destino}"))
                LoggerManager.info(f"TRUNCATE: {tabla_destino}")
        
        for i in range(num_lotes):
            inicio = i * tamaño_lote
            fin = min(inicio + tamaño_lote, len(df))
            lote = df.iloc[inicio:fin].copy()
            
            try:
                lote.to_sql(
                    name=tabla_destino,
                    con=engine_dwh,
                    if_exists='append',
                    index=False,
                    method='multi'
                )
                
                registros_lote = len(lote)
                resultados['insertados'] += registros_lote
                resultados['lotes'] += 1
                
                print(f"  Lote {i+1}/{num_lotes}: {registros_lote} registros insertados")
                LoggerManager.info(f"Lote {i+1}/{num_lotes} en {tabla_destino}: {registros_lote} registros")
                
            except Exception as e:
                resultados['errores'] += len(lote)
                LoggerManager.error(f"Error en lote {i+1} de {tabla_destino}: {str(e)}")
                print(f"  [ERROR] Lote {i+1}/{num_lotes}: {str(e)}")
        
    except Exception as e:
        LoggerManager.error(f"Error crítico en inserción de {tabla_destino}: {str(e)}")
        print(f"[ERROR CRÍTICO] {tabla_destino}: {str(e)}")
    
    return resultados


print("[OK] Funciones de transformación cargadas")

[OK] Funciones de transformación cargadas


In [ ]:
# ============================================
# ORQUESTACIÓN DE TRANSFORMACIÓN
# ============================================

print("\n" + "="*80)
print("INICIANDO TRANSFORMACIÓN - ETL STAGING -> DWH")
print("="*80 + "\n")

# NOTA: El DWH utiliza un modelo dimensional con tablas desnormalizadas.
# Las tablas intermedias (Facultad, Departamento, Programa, Curso, Docente)
# se han consolidado en las dimensiones (dim_dictado y dim_estudiante).
# Mapeo de tablas: (tabla_staging, tabla_dwh, función_transformación)

# 1. CARGAR DIMENSIÓN TIEMPO PRIMERO
print("\n" + "="*60)
print("CARGANDO DIMENSIÓN TIEMPO")
print("="*60)
stats_tiempo = cargar_dim_tiempo()
print(f"✓ Dimensión tiempo cargada: {stats_tiempo['insertados']} fechas")

transformaciones = [
    ('stg_estudiante', 'dim_estudiante', transformar_estudiante),
    ('stg_dictado', 'dim_dictado', transformar_dictado),
    ('stg_inscripcion', 'fact_inscripcion', transformar_inscripcion),
    ('stg_examen', 'fact_examen_estudiante', transformar_examen),
    ('stg_evaluacion_curso', 'fact_evaluacion_dictado', transformar_evaluacion_curso),
]

reporte_general = {}

for tabla_stg, tabla_dwh, funcion_transform in transformaciones:
    print(f"\n{'='*80}")
    print(f"Transformando: {tabla_stg} -> {tabla_dwh}")
    print(f"{'='*80}")
    
    try:
        # 1. LECTURA desde Staging
        print(f"\n[1] Leyendo datos desde {tabla_stg}...")
        df_staging = pd.read_sql(f"SELECT * FROM {tabla_stg}", con=engine_stg)
        print(f"    → {len(df_staging)} registros leídos")
        LoggerManager.info(f"Lectura {tabla_stg}: {len(df_staging)} registros")
        
        if df_staging.empty:
            print(f"    [WARN] Tabla {tabla_stg} vacía")
            LoggerManager.warning(f"Tabla vacía: {tabla_stg}")
            reporte_general[tabla_stg] = {
                'lectura': 0,
                'transformacion': {'total': 0},
                'insercion': {'insertados': 0}
            }
            continue
        
        # 2. TRANSFORMACIÓN
        print(f"\n[2] Aplicando transformaciones...")
        df_transformado, stats_transform = funcion_transform(df_staging)
        
        print(f"    → Total: {stats_transform['total']}")
        print(f"    → Válidos: {stats_transform['válidos']}")
        print(f"    → Rechazados: {stats_transform['rechazados']}")
        print(f"    → Duplicados eliminados: {stats_transform['duplicados']}")
        print(f"    → Registros finales: {len(df_transformado)}")
        
        LoggerManager.info(f"Transformación {tabla_stg}: {len(df_transformado)} registros válidos")
        
        # 3. INSERCIÓN en DWH
        print(f"\n[3] Insertando en DWH ({tabla_dwh})...")
        stats_insert = insertar_en_lotes(df_transformado, tabla_dwh, tamaño_lote=500)
        
        print(f"    → Insertados: {stats_insert['insertados']}")
        print(f"    → Errores: {stats_insert['errores']}")
        print(f"    → Lotes procesados: {stats_insert['lotes']}")
        
        LoggerManager.info(f"Inserción {tabla_dwh}: {stats_insert['insertados']} registros")
        
        # 4. VALIDACIÓN POST-INSERCIÓN
        print(f"\n[4] Validando integridad en DWH...")
        with engine_dwh.connect() as conn:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {tabla_dwh}"))
            count_final = result.fetchone()[0]
        
        status = "[OK]" if count_final == stats_insert['insertados'] else "[WARN]"
        print(f"    {status} Registros en {tabla_dwh}: {count_final}")
        
        reporte_general[tabla_stg] = {
            'lectura': len(df_staging),
            'transformacion': stats_transform,
            'insercion': stats_insert,
            'final': count_final
        }
        
        print(f"\n[OK] Transformación completada para {tabla_stg}")
        LoggerManager.info(f"COMPLETADO: {tabla_stg} -> {tabla_dwh}")
        
    except Exception as e:
        print(f"\n[ERROR] Fallo en transformación de {tabla_stg}: {str(e)}")
        LoggerManager.error(f"Error en {tabla_stg}: {str(e)}")
        reporte_general[tabla_stg] = {'error': str(e)}

print("\n" + "="*80)
print("TRANSFORMACIÓN COMPLETADA")
print("="*80)


INICIANDO TRANSFORMACIÓN - ETL STAGING -> DWH


CARGANDO DIMENSIÓN TIEMPO


2026-05-11 03:24:20 - INFO - DELETE: dim_tiempo (con FK desactivadas)
2026-05-11 03:24:20 - INFO - Lote 1/6 en dim_tiempo: 1000 registros
2026-05-11 03:24:20 - INFO - Lote 2/6 en dim_tiempo: 1000 registros


  Lote 1/6: 1000 registros insertados
  Lote 2/6: 1000 registros insertados


2026-05-11 03:24:20 - INFO - Lote 3/6 en dim_tiempo: 1000 registros
2026-05-11 03:24:20 - INFO - Lote 4/6 en dim_tiempo: 1000 registros


  Lote 3/6: 1000 registros insertados
  Lote 4/6: 1000 registros insertados


...

    → Total: 130000
    → Válidos: 130000
    → Rechazados: 0
    → Duplicados eliminados: 5041
    → Registros finales: 127433

[3] Insertando en DWH (dim_estudiante)...
[ERROR CRÍTICO] dim_estudiante: (pymysql.err.OperationalError) (1701, 'Cannot truncate a table referenced in a foreign key constraint (`dw_universidad`.`fact_examen_estudiante`, CONSTRAINT `fk_exam_estudiante`)')
[SQL: TRUNCATE TABLE dim_estudiante]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
    → Insertados: 0
    → Errores: 0
    → Lotes procesados: 0

[4] Validando integridad en DWH...
    [OK] Registros en dim_estudiante: 0

[OK] Transformación completada para stg_estudiante

Transformando: stg_dictado -> dim_dictado

[1] Leyendo datos desde stg_dictado...
    → 2261 registros leídos

[2] Aplicando transformaciones...


2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C1' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C2' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C1' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C2' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C2' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C2' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C1' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C2' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C1' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C2' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C1' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C2' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C1' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C2' a int
2026-05-11 03:24:54 - WARNING - No se pudo convertir 'C2' a int
2026-05-11 03:24:54 - WARNING - No se pu


[ERROR] Fallo en transformación de stg_dictado: 'codigo_raw'

Transformando: stg_inscripcion -> fact_inscripcion

[1] Leyendo datos desde stg_inscripcion...


2026-05-11 03:25:18 - INFO - Lectura stg_inscripcion: 1003413 registros


    → 1003413 registros leídos

[2] Aplicando transformaciones...


In [ ]:
# ============================================
# REPORTE FINAL
# ============================================

print("="*80)
print("REPORTE DE TRANSFORMACIÓN")
print("="*80)

for tabla, stats in reporte_general.items():
    if 'error' in stats:
        print(f"\n[ERROR] {tabla}: {stats['error']}")
    else:
        print(f"\n{tabla}:")
        print(f"  Lectura: {stats['lectura']:6} registros")
        print(f"  Válidos: {stats['transformacion']['válidos']:6} | Rechazados: {stats['transformacion']['rechazados']:6}")
        print(f"  Duplicados: {stats['transformacion']['duplicados']:6}")
        print(f"  Insertados: {stats['insercion']['insertados']:6} | Errores: {stats['insercion']['errores']:6}")
        print(f"  Final en DWH: {stats['final']:6} registros")

# Resumen
print(f"\n{'='*80}")
print("RESUMEN GENERAL")
print(f"{'='*80}")

total_leidos = sum(s['lectura'] for s in reporte_general.values() if 'lectura' in s)
total_válidos = sum(s['transformacion']['válidos'] for s in reporte_general.values() if 'transformacion' in s)
total_rechazados = sum(s['transformacion']['rechazados'] for s in reporte_general.values() if 'transformacion' in s)
total_duplicados = sum(s['transformacion']['duplicados'] for s in reporte_general.values() if 'transformacion' in s)
total_insertados = sum(s['insercion']['insertados'] for s in reporte_general.values() if 'insercion' in s)
total_errores = sum(s['insercion']['errores'] for s in reporte_general.values() if 'insercion' in s)

print(f"\nTotal registros leídos: {total_leidos}")
print(f"Total registros válidos: {total_válidos}")
print(f"Total registros rechazados: {total_rechazados}")
print(f"Total duplicados eliminados: {total_duplicados}")
print(f"Total registros insertados: {total_insertados}")
print(f"Total errores en inserción: {total_errores}")

porcentaje_válidos = (total_válidos / total_leidos * 100) if total_leidos > 0 else 0
porcentaje_insertados = (total_insertados / total_válidos * 100) if total_válidos > 0 else 0

print(f"\nCalidad de datos: {porcentaje_válidos:.2f}% válidos")
print(f"Tasa de inserción: {porcentaje_insertados:.2f}%")

if total_errores == 0 and total_insertados == total_válidos:
    print(f"\n[OK] TRANSFORMACIÓN EXITOSA SIN ERRORES ✓")
    LoggerManager.info("Transformación completada exitosamente")
else:
    print(f"\n[WARN] Se encontraron problemas - revisar log")
    LoggerManager.warning("Transformación completada con problemas")

print(f"\nLog guardado en: {LoggerManager.obtener_ruta_logs()}")

2026-05-02 00:20:10,324 - INFO - Transformación completada exitosamente



REPORTE DE TRANSFORMACIÓN

[ERROR] stg_facultad: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_departamento: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_programa: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_curso: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_curso_programa: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_docente: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_estudiante: (pymysql.err.OperationalError) (1049, "Unknown d